In [4]:
# 1. Ask user question (or set directly)
question = "What are recent advances in gene therapy for cancer?"  # change as needed

# 2. Query your vector store
from vector_store import VectorStore
vs = VectorStore(persist_directory="chroma_db", model_name="sentence-transformers/all-MiniLM-L6-v2")
query_results = vs.query(question, k=5)

# 3. Extract context documents
docs_for_prompt = []
for idx, doc_text in enumerate(query_results["documents"][0]):
    doc_id = query_results["ids"][0][idx]
    metadata = query_results["metadatas"][0][idx]
    docs_for_prompt.append({
        "pmid":      metadata.get("pmid", doc_id),
        "title":     metadata.get("title", "No title"),
        "abstract":  doc_text
    })

# 4. Construct prompt with context + query
context_str = ""
for d in docs_for_prompt:
    context_str += f"PMID: {d['pmid']}\nTitle: {d['title']}\nAbstract: {d['abstract']}\n\n"

prompt = f"""You are a biomedical research assistant.
User Question: {question}

Use the following reference documents to answer the question. If you cannot find enough information in the documents, respond with "Not enough data to answer."
Documents:
{context_str}

Now provide a concise, accurate answer and include citations (PMID) where appropriate.
"""

# 5. Generate response using Groq + LLaMA-3
from summarizer import LLMSummarizer
summarizer = LLMSummarizer()
answer = summarizer.generate_answer(question, docs_for_prompt)

print("🧠 Answer:")
print(answer)


🧠 Answer:
Recent advances in gene therapy for cancer include the use of exosomes as a platform for delivering therapeutic payloads, such as small molecules, RNA, or CRISPR-Cas systems, to cancer cells [PMID: 41155971]. This approach offers a highly specific and low-immunogenic alternative to viral vectors or synthetic nanoparticles. Additionally, gene therapy can introduce therapeutic genes into cancer cells to inhibit tumor growth or induce apoptosis, as seen in the use of the htsFLT01/MiRGD complex to enhance apoptosis and suppress angiogenesis in breast cancer cells [PMID: 41158839]. Furthermore, advances in T-cell immunotherapy, including the use of CRISPR-Cas9-based CAR-T optimization, have shown promise in enhancing therapeutic efficacy [PMID: 41159033]. These approaches highlight the potential of gene therapy in cancer treatment and offer promising avenues for future research and development.
